In [4]:
# -*- coding: utf-8 -*-
"""
HP ESG-D3000B Signal Generator 제어 코드
- 장비 모델: Hewlett-Packard ESG-D3000B (E4432B 아님 — 명령어 문법 다름)
- 수정: SCPI 명령어를 ESG-D3000B 전용 문법으로 전면 교체
"""

import pyvisa
import re
import time

E4432B_address = "GPIB43::26::INSTR"  # 실제 장비는 ESG-D3000B

def open_resource(resource_id):
    # VISA 리소스 매니저로 장비 연결
    rm = pyvisa.ResourceManager()
    return rm.open_resource(resource_id)

def clear_errors(instrument):
    # 에러 큐를 비움 — 코드 실행 전 반드시 호출
    instrument.write("*CLS")
    print("에러 큐 초기화 완료 (*CLS 전송)")

def rf_on(instrument):
    # RF 출력 켜기 — ESG-D3000B 전용 명령어
    instrument.write(":OUTP:STAT ON")
    time.sleep(0.2)
    print("RF 출력: ON")

def rf_off(instrument):
    # RF 출력 끄기 — ESG-D3000B 전용 명령어
    instrument.write(":OUTP:STAT OFF")
    time.sleep(0.2)
    print("RF 출력: OFF")

def present_freq(instrument):
    # 현재 주파수 읽기 — ESG-D3000B는 :FREQ:CW? 사용
    try:
        response = instrument.query(":FREQ:CW?")
        # 응답 예시: '+6.5000000000000E+07'
        match = re.search(r'([+-]?\d+\.?\d*(?:E[+-]?\d+)?)', response, re.IGNORECASE)
        if match:
            return float(match.group(1))
        else:
            raise ValueError(f"예상치 못한 응답 형식: {response}")
    except Exception as e:
        print(f"주파수 읽기 오류: {e}")
        return None

def windup_freq(instrument, target_freq, step_size, delay):
    # 현재 주파수 → target_freq 까지 step_size(Hz)씩 단계적으로 변경
    # ESG-D3000B 주파수 범위: 250 kHz ~ 3 GHz
    current = present_freq(instrument)
    if current is None:
        print("주파수 읽기 실패 — wind-up 중단")
        return

    step = abs(step_size) if current < target_freq else -abs(step_size)

    while abs(current - target_freq) > abs(step) / 2:
        new_freq = current + step
        # 목표값을 넘지 않도록 클램프
        if (step > 0 and new_freq > target_freq) or \
           (step < 0 and new_freq < target_freq):
            new_freq = target_freq
        # ESG-D3000B 주파수 설정 명령어: :FREQ:CW
        instrument.write(f":FREQ:CW {new_freq:.6f}HZ")
        print(f"  주파수 wind-up: {new_freq:.3e} Hz")
        time.sleep(delay)
        current = present_freq(instrument)
        if current is None:
            print("주파수 읽기 실패 — wind-up 중단")
            return

    # 최종값 정확히 설정
    instrument.write(f":FREQ:CW {target_freq:.6f}HZ")
    print(f"최종 주파수 설정 완료: {target_freq:.3e} Hz")

def present_Prf(instrument):
    # 현재 RF 파워 읽기 — ESG-D3000B는 :POW? 사용
    try:
        response = instrument.query(":POW?")
        # 응답 예시: '+1.00000000E+000'
        match = re.search(r'([+-]?\d+\.?\d*(?:E[+-]?\d+)?)', response, re.IGNORECASE)
        if match:
            return float(match.group(1))
        else:
            raise ValueError(f"예상치 못한 응답 형식: {response}")
    except Exception as e:
        print(f"파워 읽기 오류: {e}")
        return None

def windup_Prf(instrument, target_prf, step_size, delay):
    # 현재 파워 → target_prf(dBm)까지 step_size(dBm)씩 단계적으로 변경
    # ESG-D3000B 파워 범위: -136 ~ +25 dBm (모델에 따라 상이)
    current = present_Prf(instrument)
    if current is None:
        print("파워 읽기 실패 — wind-up 중단")
        return

    step = abs(step_size) if current < target_prf else -abs(step_size)

    while abs(current - target_prf) > abs(step) / 2:
        new_prf = current + step
        # 목표값을 넘지 않도록 클램프
        if (step > 0 and new_prf > target_prf) or \
           (step < 0 and new_prf < target_prf):
            new_prf = target_prf
        # ESG-D3000B 파워 설정 명령어: :POW
        instrument.write(f":POW {new_prf:.3f}DBM")
        print(f"  파워 wind-up: {new_prf:.3f} dBm")
        time.sleep(delay)
        current = present_Prf(instrument)
        if current is None:
            print("파워 읽기 실패 — wind-up 중단")
            return

    # 최종값 정확히 설정
    instrument.write(f":POW {target_prf:.3f}DBM")
    print(f"최종 파워 설정 완료: {target_prf:.3f} dBm")

def main():
    instrument = open_resource(E4432B_address)

    # 시작 전 에러 큐 초기화 — 이전 에러가 남아있으면 ERR 표시됨
    clear_errors(instrument)

    # --- 목표값 설정 ---
    target_prf  = -20       # [dBm]  목표 RF 파워
    step_prf    = 1       # [dBm]  파워 step 크기 (1 dBm씩 변경)
    delay_prf   = 0.2     # [sec]

    target_freq = 65e6    # [Hz]   목표 주파수 = 65 MHz
    step_freq   = 1e6     # [Hz]   주파수 step 크기 (1 MHz씩 변경)
    delay_freq  = 0.2     # [sec]

    # 1단계: 파워 wind-up
    windup_Prf(instrument, target_prf, step_prf, delay_prf)

    # 2단계: 주파수 wind-up
    windup_freq(instrument, target_freq, step_freq, delay_freq)

    # 3단계: RF 출력 켜기
    rf_on(instrument)

    # 최종 상태 확인
    final_prf  = present_Prf(instrument)
    final_freq = present_freq(instrument)

    if final_prf  is not None: print(f"\n설정된 파워  : {final_prf:.3f} dBm")
    if final_freq is not None: print(f"설정된 주파수: {final_freq:.3e} Hz")

    instrument.close()

if __name__ == "__main__":
    main()

에러 큐 초기화 완료 (*CLS 전송)
  파워 wind-up: -11.000 dBm
  파워 wind-up: -12.000 dBm
  파워 wind-up: -13.000 dBm
  파워 wind-up: -14.000 dBm
  파워 wind-up: -15.000 dBm
  파워 wind-up: -16.000 dBm
  파워 wind-up: -17.000 dBm
  파워 wind-up: -18.000 dBm
  파워 wind-up: -19.000 dBm
  파워 wind-up: -20.000 dBm
최종 파워 설정 완료: -20.000 dBm
최종 주파수 설정 완료: 6.500e+07 Hz
RF 출력: ON

설정된 파워  : -20.000 dBm
설정된 주파수: 6.500e+07 Hz
